In [2]:
# Vector Data impimentation
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [3]:
loader = PyPDFLoader("../test_documents/attention.pdf")

In [4]:
document = loader.load()

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
documents =  text_splitter.split_documents(document)
vectors = FAISS.from_documents(documents, embeddings)

In [6]:
retriver = vectors.as_retriever()

In [7]:
from langchain_core.tools import tool

@tool
def database_search(query: str) -> str:
    """Search for information about the research paper.
    You should use this tool to get the information about the relevant research paper."""
    docs = retriver.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [8]:
parse_agent_Tools = [database_search]

# Agent Implimentation 

# Document Parse Agent

In [9]:
## Agents
from langchain.agents import create_agent
from langchain.tools import tool


from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
load_dotenv()


True

In [10]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
parse_llm_with_tools = llm.bind_tools(parse_agent_Tools)

In [138]:
parse_agent_prompt = """You are the Research Document Parsing Agent. 
First Use the given retriver_tool Tool to get research paper context and  You Read the given  research paper  Fully and divide them into Abstract, Method, Math, Experiments, Results, Youtube.
Rules:
- Do NOT wait for user input or paper name
- Go directly to the vector database and extract
- Do NOT ask questions
- Use the document retrieval tool to read the paper
- Be concise but complete
- GIVE THE FINAL RESULT IN JSON KEY VALUE PAIRS JUST JSON NO JARGON NO STRING
- Include a search query for YOUTUBE to learn more aboute the content on research paper. Don't search youtube just provide query only
"""

In [119]:
parse_agent = create_agent(
    model=parse_llm_with_tools,  # or any other supported model identifier string
    tools=parse_agent_Tools,
    system_prompt=parse_agent_prompt
)

In [126]:
inputs = {"messages": [{"role": "user", "content": "Analyze the given research paper"}]}
result_parse_agent=parse_agent.invoke(inputs)
print(result_parse_agent["messages"][-1].content)

{
  "Abstract": "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly fewer computational resources.",
  "Method": "The Transformer consists of an encoder stack of N identical layers, each with multi-head self‑attention and a position‑wise feed‑forward network, followed by residual connections and layer normalization. The decoder stack similarly uses masked multi‑head self‑attention, encoder‑decoder attention, and feed‑forward layers. Positional encodings are added to input embeddings to provide sequenc

In [70]:
result_parse_agent["messages"][-1].content

'{"abstract":"The Transformer is a sequence‑to‑sequence model that replaces recurrent and convolutional layers with multi‑head self‑attention. It achieves state‑of‑the‑art performance on machine translation while being highly parallelizable and faster to train.","method":"Encoder and decoder each consist of N identical layers. Each layer has a multi‑head self‑attention sub‑layer followed by a position‑wise feed‑forward network. The encoder uses self‑attention over the entire input; the decoder uses masked self‑attention and encoder‑decoder attention. Residual connections and layer‑normalization surround each sub‑layer. Positional encodings (sinusoidal or learned) are added to token embeddings.","math":"Scaled dot‑product attention:  Attention(Q,K,V)=softmax(QKᵀ/√d_k)V. Multi‑head attention splits Q,K,V into h heads, applies attention in parallel, then concatenates and projects. Feed‑forward network: FFN(x)=max(0,xW₁+b₁)W₂+b₂ with d_model=512, d_ff=2048. Dropout and label smoothing are 

In [71]:
result_parse_agent

{'messages': [HumanMessage(content='Analyze the given research paper', additional_kwargs={}, response_metadata={}, id='6fca3b5b-47cf-4948-bfdc-613389e45f28'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the database_search tool to retrieve the research paper. But we don\'t have a query. The instruction says "Do NOT wait for user input or paper name". So we need to decide a query. Maybe we can search for a generic paper? But we need to produce JSON with sections. We need to pick a paper. The instruction: "First Use the given retriver_tool Tool to get research paper context and You Read the given research paper Fully and divide them into Abstract, Method, Math, Experiments, Results." But we don\'t have a paper name. Maybe we can search for a popular paper. But the instruction says "Do NOT wait for user input or paper name". So we must choose a paper. Perhaps we can search for "Attention Is All You Need" or "BERT: Pre-training of Deep Bidirectional Tran

# Teaching Agent

In [15]:
teaching_agent_prompt = """You are the Teaching Assistant Agent. 
Read the given context carefully and explain the key concepts, Architecture ideas , algorithm intuition using 
simple language, Analogies and step by step reasoning so even a complete beginner could understand.Use the tools only if you want.
Compress the result into bullet points.
Keep only essential facts, findings, and conclusions.
"""

In [16]:
wiki_api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki_tool = WikipediaQueryRun(api_wrapper= wiki_api_wrapper)

In [17]:
llm = ChatGroq(
    model="openai/gpt-oss-safeguard-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
teach_tools = [wiki_tool]
teach_llm=llm.bind_tools(teach_tools)

In [18]:
teach_agent = create_agent(
    model=teach_llm,  # or any other supported model identifier string
    tools=teach_tools,
    system_prompt=teaching_agent_prompt
)

In [19]:
teach_agent_input = {"messages": [{"role": "user", "content": "Teach me this research paper context"+ "context - "+str(result_parse_agent["messages"][-1].content)}]}
teach_agent_results=teach_agent.invoke(teach_agent_input)
print(teach_agent_results["messages"][-1].content)

**What the paper is about**  
- Introduces the **Transformer** – a new way to translate text that uses *only* attention, no RNNs or CNNs.  
- It gets better translation scores (BLEU) on big English‑German and English‑French datasets and trains *much* faster.

---

### 1. High‑level architecture  
| Part | What it does | Analogy |
|------|--------------|---------|
| **Encoder** | Reads the whole source sentence at once, producing a “context” for every word. | A group of people all looking at the same picture and writing down what they see. |
| **Decoder** | Generates the target sentence one word at a time, but can look back at the encoder’s context. | A writer who can ask the picture‑group for help while writing. |
| **Layers** | 6 identical encoder layers + 6 decoder layers (you can make more). | 6 rounds of discussion for each side. |

Each layer has three sub‑steps (except the decoder’s first one):

1. **Self‑attention** – every word looks at every other word in the same sentence.  


In [20]:
teach_agent_results

{'messages': [HumanMessage(content='Teach me this research paper contextcontext - **Abstract**  \nThe paper introduces the *Transformer*, a sequence‑to‑sequence model that relies exclusively on attention mechanisms, eliminating recurrent and convolutional layers. The architecture achieves state‑of‑the‑art BLEU scores on WMT 2014 English‑German and English‑French translation tasks while training far faster and allowing greater parallelism.\n\n---\n\n**Method**  \n1. **Encoder–Decoder Stack**  \n   * Six identical encoder layers and six decoder layers (N\u202f=\u202f6).  \n   * Each encoder layer:  \n     * Multi‑head self‑attention (queries, keys, values all from encoder output).  \n     * Position‑wise feed‑forward network (FFN).  \n   * Each decoder layer:  \n     * Masked multi‑head self‑attention (preventing future‑token access).  \n     * Encoder–decoder multi‑head attention (queries from decoder, keys/values from encoder).  \n     * Position‑wise FFN.  \n\n2. **Attention Mechanism

# Mathematical Reasoning Agent

In [21]:
Math_agent_prompt = """You are the Mathematical Reasoning Agent. 
Read the given research paper context carefully, extract all the mathematical formulas and explain , analyze equations, Loss functions, 
optimization methods.
Explain what each equation represents, why it needed and assumptions made so even a complete beginner could understand.
present the results in a human-readable format. 
Compress the result into bullet points.
Keep only essential facts, findings, and conclusions.
Use the provided tools ONLY if you need.

"""

In [22]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
math_tools = [wiki_tool]
math_llm=llm.bind_tools(math_tools)

In [23]:
math_agent = create_agent(
    model=math_llm,  # or any other supported model identifier string
    tools=math_tools,
    system_prompt=Math_agent_prompt
)

In [24]:
math_agent_input = {"messages": [{"role": "user", "content": "explain all the mathematical terms in this research paper"+ "context - "+str(result_parse_agent["messages"][-1].content)}]}
math_agent_results=teach_agent.invoke(math_agent_input)
print(math_agent_results["messages"][-1].content)

**Key mathematical ideas in the Transformer paper – explained simply**

| Concept | What it is | Why it matters | Simple analogy / step‑by‑step |
|---------|------------|----------------|------------------------------|
| **Scaled dot‑product attention**<br> \(\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V\) | A way to let each word “look at” every other word and decide how much it should listen. <br>• **Q** (query) – what we’re asking for.<br>• **K** (key) – what each word can offer.<br>• **V** (value) – the actual information to pass on. | Gives a weighted sum of all values, where weights come from how similar Q and K are. The scaling \(\sqrt{d_k}\) keeps the numbers from blowing up when the vectors are long. | Think of a classroom: a student (query) asks a question, each other student (key) has an answer (value). The teacher (softmax) decides how much to trust each answer based on how relevant it is. |
| **Softmax** | Turns raw scores into probabil

In [25]:
math_agent_results

{'messages': [HumanMessage(content='explain all the mathematical terms in this research papercontext - **Abstract**  \nThe paper introduces the *Transformer*, a sequence‑to‑sequence model that relies exclusively on attention mechanisms, eliminating recurrent and convolutional layers. The architecture achieves state‑of‑the‑art BLEU scores on WMT 2014 English‑German and English‑French translation tasks while training far faster and allowing greater parallelism.\n\n---\n\n**Method**  \n1. **Encoder–Decoder Stack**  \n   * Six identical encoder layers and six decoder layers (N\u202f=\u202f6).  \n   * Each encoder layer:  \n     * Multi‑head self‑attention (queries, keys, values all from encoder output).  \n     * Position‑wise feed‑forward network (FFN).  \n   * Each decoder layer:  \n     * Masked multi‑head self‑attention (preventing future‑token access).  \n     * Encoder–decoder multi‑head attention (queries from decoder, keys/values from encoder).  \n     * Position‑wise FFN.  \n\n2. 

# Experimental Analysis Agent

In [26]:
experimental_analysis_agent_prompt = """You are the Experimental Analysis Agent and Empirical reviewer. 
Read the given context carefully and explain / Reviews Datasets used, Experimental setup, Baselines, Evaluation metrics.
Determines whether experiments support claims, If comparisons are fair. Conpare them with other related research papers.
Compress the result into bullet points.
Keep only essential facts, findings, and conclusions.
Use the given tools only if needed. Give me exacly a final answer don't think too much.
"""

In [27]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv_tool = ArxivQueryRun(api_wrapper= arxiv_wrapper)

In [28]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
experimental_analysis_agent_tools = [arxiv_tool,wiki_tool]
experimental_analysis_agent_llm=llm.bind_tools(experimental_analysis_agent_tools)

In [29]:
experimental_analysis_agent = create_agent(
    model=experimental_analysis_agent_llm,  # or any other supported model identifier string
    tools=experimental_analysis_agent_tools,
    system_prompt=experimental_analysis_agent_prompt
)

In [30]:
experimental_analysis_agent_input = {"messages": [{"role": "user", "content": "experiment analysis on the given research paper"+ "context - "+str(result_parse_agent["messages"][-1].content)}]}
experimental_analysis_agent_results=experimental_analysis_agent.invoke(experimental_analysis_agent_input)
print(experimental_analysis_agent_results["messages"][-1].content)

**Paper:** “Attention Is All You Need” (Vaswani et al., 2017) – Transformer  

---

### Datasets & Tasks
- **WMT 2014 English‑German (EN‑DE)** – standard MT benchmark; evaluation on *newstest2014*.
- **WMT 2014 English‑French (EN‑FR)** – same split, evaluation on *newstest2014*.
- Both corpora are pre‑processed with byte‑pair encoding (BPE) (≈ 32 k sub‑word units) as in prior work, ensuring a fair comparison with LSTM baselines.

### Experimental Setup
| Component | Details |
|-----------|---------|
| **Model variants** | *Base* (d_model = 512, h = 8, d_ff = 2048, N = 6) and *Big* (d_model = 1024, h = 16, d_ff = 4096). |
| **Training** | 1 M steps, batch ≈ 25 k tokens, 4 × NVIDIA Tesla K80 GPUs, ~1 h/step. |
| **Optimizer** | Adam (β₁ = 0.9, β₂ = 0.98, ε = 10⁻⁹) with the “warm‑up” learning‑rate schedule (4000 warm‑up steps). |
| **Regularisation** | Dropout = 0.1 on all sub‑layers; label smoothing (ε = 0.1). |
| **Positional encoding** | Fixed sinusoidal; ablation shows learned embeddi

In [31]:
experimental_analysis_agent_results

{'messages': [HumanMessage(content='experiment analysis on the given research papercontext - **Abstract**  \nThe paper introduces the *Transformer*, a sequence‑to‑sequence model that relies exclusively on attention mechanisms, eliminating recurrent and convolutional layers. The architecture achieves state‑of‑the‑art BLEU scores on WMT 2014 English‑German and English‑French translation tasks while training far faster and allowing greater parallelism.\n\n---\n\n**Method**  \n1. **Encoder–Decoder Stack**  \n   * Six identical encoder layers and six decoder layers (N\u202f=\u202f6).  \n   * Each encoder layer:  \n     * Multi‑head self‑attention (queries, keys, values all from encoder output).  \n     * Position‑wise feed‑forward network (FFN).  \n   * Each decoder layer:  \n     * Masked multi‑head self‑attention (preventing future‑token access).  \n     * Encoder–decoder multi‑head attention (queries from decoder, keys/values from encoder).  \n     * Position‑wise FFN.  \n\n2. **Attentio

# Testing Multi Agent Architecture

In [36]:
print(result_parse_agent["messages"][-1].content)

**Abstract**  
The paper introduces the *Transformer*, a sequence‑to‑sequence model that relies exclusively on attention mechanisms, eliminating recurrent and convolutional layers. The architecture achieves state‑of‑the‑art BLEU scores on WMT 2014 English‑German and English‑French translation tasks while training far faster and allowing greater parallelism.

---

**Method**  
1. **Encoder–Decoder Stack**  
   * Six identical encoder layers and six decoder layers (N = 6).  
   * Each encoder layer:  
     * Multi‑head self‑attention (queries, keys, values all from encoder output).  
     * Position‑wise feed‑forward network (FFN).  
   * Each decoder layer:  
     * Masked multi‑head self‑attention (preventing future‑token access).  
     * Encoder–decoder multi‑head attention (queries from decoder, keys/values from encoder).  
     * Position‑wise FFN.  

2. **Attention Mechanism**  
   * **Scaled dot‑product attention**:  
     \[
     \text{Attention}(Q,K,V)=\text{softmax}\!\left(\fr

In [82]:
def filter_context(parsed_content):
    """
    Splits the parsed research paper into sections for each sub-agent
    """
    # Example: You may parse parsed_content string into a dict or sections
    # Adjust according to your parse_agent output format
    context_dict = {
        "teach": parsed_content.get("abstract", "") + "\n" + parsed_content.get("Method", ""),
        "math": parsed_content.get("math", ""),
        "experiment": parsed_content.get("experiments", "") + "\n" + parsed_content.get("Results", "")
    }
    return context_dict

In [83]:
parsed_content_info = result_parse_agent["messages"][-1].content

In [84]:
parsed_content_info

'{"abstract":"The Transformer is a sequence‑to‑sequence model that replaces recurrent and convolutional layers with multi‑head self‑attention. It achieves state‑of‑the‑art performance on machine translation while being highly parallelizable and faster to train.","method":"Encoder and decoder each consist of N identical layers. Each layer has a multi‑head self‑attention sub‑layer followed by a position‑wise feed‑forward network. The encoder uses self‑attention over the entire input; the decoder uses masked self‑attention and encoder‑decoder attention. Residual connections and layer‑normalization surround each sub‑layer. Positional encodings (sinusoidal or learned) are added to token embeddings.","math":"Scaled dot‑product attention:  Attention(Q,K,V)=softmax(QKᵀ/√d_k)V. Multi‑head attention splits Q,K,V into h heads, applies attention in parallel, then concatenates and projects. Feed‑forward network: FFN(x)=max(0,xW₁+b₁)W₂+b₂ with d_model=512, d_ff=2048. Dropout and label smoothing are 

In [85]:
import json
import re

def clean_json(markdown_json_str):
    """
    Cleans a JSON string that may have markdown code block syntax and returns a Python dictionary.
    
    Args:
        markdown_json_str (str): JSON string possibly wrapped in triple backticks.
        
    Returns:
        dict: Parsed JSON as a Python dictionary.
    """
    # Remove markdown code block backticks and optional language tag (like ```json)
    cleaned_str = re.sub(r'^```[a-z]*\n|```$', '', markdown_json_str.strip(), flags=re.IGNORECASE)
    
    # Parse the JSON string into a Python dictionary
    try:
        data = json.loads(cleaned_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON: {e}")
    
    return data

In [86]:
cleaned_version = clean_json(parsed_content_info)

In [87]:
filter_context(cleaned_version)

{'teach': 'The Transformer is a sequence‑to‑sequence model that replaces recurrent and convolutional layers with multi‑head self‑attention. It achieves state‑of‑the‑art performance on machine translation while being highly parallelizable and faster to train.\n',
 'math': 'Scaled dot‑product attention:  Attention(Q,K,V)=softmax(QKᵀ/√d_k)V. Multi‑head attention splits Q,K,V into h heads, applies attention in parallel, then concatenates and projects. Feed‑forward network: FFN(x)=max(0,xW₁+b₁)W₂+b₂ with d_model=512, d_ff=2048. Dropout and label smoothing are used.',
 'experiment': 'Trained on WMT 2014 English‑German (≈4.5M sentence pairs) and WMT 2014 English‑French (≈36M pairs). Baselines: 5‑layer LSTM encoder‑decoder with attention. Models: base (N=6, d_model=512, h=8) and big (N=6, d_model=1024, h=16). Training used Adam optimizer, learning‑rate schedule with warm‑up, batch size 4096 tokens. Evaluation via BLEU.\n'}

In [ ]:
parse_agent_Tools = [database_search]

# Full parser pipeline

In [139]:
def full_parser_pipeline(path):
    #Doc indexing
    print("document indexing.....")
    loader = PyPDFLoader(path)
    document = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
    documents =  text_splitter.split_documents(document)
    vectors = FAISS.from_documents(documents, embeddings)
    retriver = vectors.as_retriever()

    print("Retriever as a tool.....")
    
    @tool
    def database_search(query: str) -> str:
        """Search for information about the research paper.
        You should use this tool to get the information about the relevant research paper."""
        docs = retriver.invoke(query)
        return "\n\n".join([doc.page_content for doc in docs])

    parse_agent_Tools = [database_search]

    llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
    )
    parse_llm_with_tools = llm.bind_tools(parse_agent_Tools)


    parse_agent = create_agent(
    model=parse_llm_with_tools,  # or any other supported model identifier string
    tools=parse_agent_Tools,
    system_prompt=parse_agent_prompt
    )
    print("Agent execution.....")
    inputs = {"messages": [{"role": "user", "content": "Analyze the given research paper"}]}
    result_parse_agent=parse_agent.invoke(inputs)
    retrieved_cont_info = result_parse_agent["messages"][-1].content
    print("Agent workflow COMPLETED")
    return retrieved_cont_info;
    

In [140]:
full_jsoned_content = full_parser_pipeline("../test_documents/ALBERT.pdf")

document indexing.....
Retriever as a tool.....


TypeError: 'YouTubeSearchTool' object is not callable

In [129]:
full_jsoned_content=clean_json(full_jsoned_content)

In [132]:
full_jsoned_content['Youtube']

'Attention Is All You Need transformer tutorial'

In [107]:
full_jsoned_content['Abstract']

'ALBERT (A Lite BERT) introduces two parameter‑reduction techniques—factorized embedding parameterization and cross‑layer parameter sharing—plus a novel self‑supervised sentence‑order prediction (SOP) loss. These changes dramatically shrink model size (up to 18× fewer parameters than BERT‑large) while preserving or improving performance. The resulting models achieve new state‑of‑the‑art results on GLUE, RACE, and SQuAD benchmarks with faster training and better generalization.'

In [108]:
teach_agent_input = {"messages": [{"role": "user", "content": "Teach me this research paper context"+ "context - "+str(full_jsoned_content['Abstract'])}]}
teach_agent_results=teach_agent.invoke(teach_agent_input)
print(teach_agent_results["messages"][-1].content)

**ALBERT (A Lite BERT)** – a lighter, faster, and still‑powerful language model

---

### 1. Why “Lite” BERT?
- **BERT** (Bidirectional Encoder Representations from Transformers) is great but huge – ~110 M parameters for BERT‑base, ~340 M for BERT‑large.
- Training such a big model is expensive (time, memory, energy).
- ALBERT’s goal: *cut the size dramatically* while keeping or improving accuracy.

---

### 2. Two key tricks that shrink the model

| Trick | What it does | Analogy | Result |
|-------|--------------|---------|--------|
| **Factorized embedding parameterization** | Split the word‑embedding matrix into two smaller matrices: one for *vocabulary → hidden* and one for *hidden → vocabulary*. | Think of a big dictionary that’s split into “word‑to‑concept” and “concept‑to‑word” parts, each smaller. | Reduces embedding parameters from *V × H* to *V × d + H × d* (d ≪ H). |
| **Cross‑layer parameter sharing** | Use the same weights for every transformer layer instead of having a f

In [111]:
math_agent_input = {"messages": [{"role": "user", "content": "explain all the mathematical terms in this research paper"+ "context - "+str(full_jsoned_content['Math'])}]}
math_agent_results=math_agent.invoke(math_agent_input)
print(math_agent_results["messages"][-1].content)

**Key Mathematical Concepts in the Paper (Explained for Beginners)**  

---

### 1. Factorized Embedding  

- **Formula:**  
  \[
  E = E_1 \times E_2 ,\qquad 
  E_1 \in \mathbb{R}^{V\times E},\; 
  E_2 \in \mathbb{R}^{E\times H}
  \]

- **What it means:**  
  - **\(V\)** = size of the vocabulary (number of distinct tokens/words).  
  - **\(E\)** = a chosen intermediate dimension (often much smaller than \(H\)).  
  - **\(H\)** = final hidden‑size used by the model (e.g., 768 in BERT‑base).  
  - **\(E_1\)** maps each token to an **\(E\)-dimensional** vector (a “low‑rank” representation).  
  - **\(E_2\)** then projects that \(E\)-dimensional vector up to the full hidden size **\(H\)**.  
  - Multiplying the two matrices (`E1 × E2`) yields the usual embedding matrix **\(E\)** of shape **\(V \times H\)**, but it is stored as two smaller matrices.

- **Why it is used:**  
  - **Parameter efficiency:** Instead of learning a huge \(V \times H\) matrix (which can be millions of parameters),

In [116]:
experimental_analysis_agent_input = {"messages": [{"role": "user", "content": "experiment analysis on the given research paper"+ "context - "+str(full_jsoned_content['Experiments'])}]}
experimental_analysis_agent_results=experimental_analysis_agent.invoke(experimental_analysis_agent_input)
print(experimental_analysis_agent_results["messages"][-1].content)

**Datasets**  
- **Pre‑training corpus**: 16 GB combined Wikipedia + BookCorpus (same data used for BERT).  
- **Down‑stream evaluation**:  
  - **GLUE benchmark** – MNLI, QQP, QNLI, SST‑2, CoLA, STS‑B, MRPC, RTE.  
  - **RACE** – multiple‑choice reading‑comprehension dataset.  
  - **SQuAD v1.1 / v2.0** – span‑based QA (exact‑match & F1).  

**Experimental Setup**  
- Pre‑training objective: masked language modeling (MLM) identical to BERT + “Sentence Order Prediction” (SOP) as in ALBERT.  
- Model variants incorporate:  
  - Factorized embedding parameterization.  
  - Parameter sharing across Transformer layers.  
  - SOP loss.  
- Training hyper‑parameters (batch size, learning rate, steps) match those reported for BERT‑base/large to ensure a fair comparison.  
- Evaluation follows standard GLUE, RACE, and SQuAD protocols (single‑model, no ensembling).  

**Baselines**  
- **BERT‑base** (≈110 M params) and **BERT‑large** (≈340 M params).  
- Recent compact models (e.g., ALBERT‑base

In [117]:
from langchain_community.tools import YouTubeSearchTool

In [133]:
tool = YouTubeSearchTool()

In [135]:
tool.run(full_jsoned_content['Youtube'])

"['https://www.youtube.com/watch?v=TQQlZhbC5ps&pp=ygUuQXR0ZW50aW9uIElzIEFsbCBZb3UgTmVlZCB0cmFuc2Zvcm1lciB0dXRvcmlhbA%3D%3D', 'https://www.youtube.com/watch?v=bCz4OMemCcA&pp=ygUuQXR0ZW50aW9uIElzIEFsbCBZb3UgTmVlZCB0cmFuc2Zvcm1lciB0dXRvcmlhbA%3D%3D']"